# Gemma 4 Developer Agent — Phase 0 checks, EXP-000 gold control, baseline eval (4x L4)

One notebook, three switches (`RUN_PHASE0_CHECKS`, `RUN_GOLD_CONTROL`, `RUN_AGENT`). It mirrors the
organizers' Getting-Started recipe (wheelhouse install, `VllmServer`, `swegemma.evaluate.Evaluator` with the
subprocess sandbox) and writes everything under `/kaggle/working/runs/<EXP_ID>/` in the layout that
`tools/analyze_traces.py` and `tools/estimate_budget.py` read.

**Attach as inputs** (Add Input): competition `gemma-4-developer-agent`, dataset
`metric/gemma-4-developer-agent-wheelhouse`, model `google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2`.
**Accelerator**: NVIDIA L4 x4 (competition-attached notebooks only). **Internet**: off.
The submission zip comes either from a private Kaggle dataset you upload (`SUBMISSION_ZIP`) or embedded as
base64 by `python tools/build_notebook.py --embed dist/submission.zip`.

**Status: UNTESTED on Kaggle.** This file was written without access to kaggle.com, the wheelhouse or the
harness source. Cells marked `# VERIFY` use API names taken from a community mirror of the starter notebook
and may need adjusting against the real `getting-started-gemma-4-developer-agent` notebook. Read the
starter notebook first and diff the server/evaluator cells.

Edit the parameters in the next cell, save a notebook *version* per experiment, then download
`/kaggle/working/runs/<EXP_ID>.zip` and run `python tools/analyze_traces.py runs/<EXP_ID>`.

In [ ]:
# ---------------------------------------------------------------- parameters (edit me)
EXP_ID = "EXP-025-editfix-dev48-armA-baseline"   # overridden by --set
RUN_PHASE0_CHECKS = True          # Q1/Q2/Q3/Q7/Q9 answers + HARNESS_README copy + sample_submission dump
RUN_GOLD_CONTROL = False   # overridden by --set
RUN_AGENT = True   # overridden by --set
MEASURE_DECODE_SPEED = True   # overridden by --set

# Task selection: explicit ids win; otherwise the first N_TASKS of the (optionally repo-filtered) dataset.
TASK_IDS = ["fastapi_11355", "fastapi_13207", "fastapi_13537", "fastapi_13786", "fastapi_14262", "fastapi_14297", "fastapi_14301", "fastapi_14360", "fastapi_14448", "fastapi_14459", "fastapi_14463", "fastapi_14512", "fastapi_14583", "fastapi_14605", "fastapi_14609", "fastapi_14873", "fastapi_14953", "fastapi_14962", "fastapi_14964", "fastapi_15030", "fastapi_15280", "fastapi_15589", "fastapi_15661", "fastapi_15800", "fastapi_5624", "fastapi_9753", "httpx_3672", "requests_6644", "requests_7205", "requests_7309", "requests_7315", "requests_7427", "rich_3006", "rich_3061", "rich_3063", "rich_3067", "rich_3105", "rich_3180", "rich_3468", "rich_3480", "rich_3521", "rich_3676", "rich_3905", "rich_3934", "rich_3935", "rich_3953", "rich_4070", "rich_4075"]   # overridden by --set
N_TASKS = 8
REPO_FILTER = ""                  # e.g. "Textualize/rich"; "" = all repos
EXCLUDE_IDS = []                  # known-unsound tasks from a previous gold control (experiments/README.md Q5)

# Where the submission comes from (first that exists wins). Upload dist/submission.zip as a private dataset,
# or embed it: python tools/build_notebook.py --embed dist/submission.zip
SUBMISSION_ZIP = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/gemma4-sub-v1/submission.zip"   # VERIFY path
SUBMISSION_B64 = "UEsDBBQAAAAIAAAAIVz+GJYZFAMAADIFAAAKAAAAYWdlbnQueWFtbFVU247bNhB911dMk4d4gZW99m4NxHlKETcp0DSpNwiaJ2Mkji3CvKgktVp/Uf+jX9ZDajdxAcMSKXLmnDln5iXtvE/ER3GJDj5QHBqrY9Te0cOSZn3wtk+1d+Z8TVG7o5Hp8DU5T6y4TxLKezxpY+LVvHpJ9x33gmjG+DFS6oQ+vN39sb2/3++2b9993JLVISAXO4VTQ6DWWzs4nc4X6SOSG3Zzq+jff27nd/TnqgT/ut399uu3GVYbUvpwQHwd6aALMNYuJlKceBF4XES2vZH9j6CLgn1+ZmvIu1YKuHw8SiKEUX50xrMSdZlqiVTJe0OOrURqBLzIDsjEfS8cSLsSCDR64KiTtkJfcGEnRx1TONPsGKRHuU71RX3HTsSAU466oTjKvvVKQmXxbzZ0FGu5vqtvl02tU/03p3q84+W6bhMR/b8QE1GyjAr2YgzpRKNOHTFBwQeNsHiRg3588/QEDw5CQKfBQVFzLhQ6Dk5irJTEFl8ScG5oJ9GbB9zwTui9Th+GBsWKgzwT/3xOHQiBo486eRDmRIvRh1PsGVV+gmK105bNNT1I0AeNrD2ntptXWbUwtFO2n7RrzaCEJu/FxffKwAsV9JPAKe+4BC3z86CPF9emjTiJD8cWtausX9xURDWFwe2z42C/aS2s9tlAZSVKpx+rMWjk+r48StrHxGmIZVnETPvCosqabP/6XN8s17S6Wa3r5U29Wm1KgW7pGLjvioumwu+2Hz993b6jWbEvftufX+FLYzhX4ZoSn8Sh4CSP6DDHhiTLCM/CMTnVF4QN7E71Lc1+/4Vu5qu7q9JRZXP5+nl3dUX90Bgdu6zy4JTJFvYQRElrMhR55DaZon8UWj+DRCjnS6oL7G8Knae8SdgCNHqGjG8BEZWNyJbyIR2me3XpHCg+NfXUPNP39fQNrooTp0+YBTnapDzMqIhNlucMsNYnMMBNCyj0Io8k0mj/0VPurphHScGMirrjC4wrn0+fYQh06hSyZGkGBR1LbQELHZPQm2OXJ0gjuJoN0XaQQvIZNjbfx4TDlDjNq/8AUEsDBBQAAAAIAAAAIVwBmqEaNgYAAC4NAAAVAAAAY29uZmlncy9zYW1wbGluZy55YW1shVfbbttGEH3PVwxiBJEAkbrasWWkgJMwjQFfUsVIGhSFsCJX0kJLLru7lKw+5Hf6H/2ynllSsuOgimBABrk7Zy5nzoyOaCELaYWX09QUXhaev+dqQXNjSa6l3ZLAEU+tUosizjP6959RPCTnhfWqWNBa6Eq6dvzsiC60NhuZ0VxJnTkqpaUPF5Ob5NOn6SS5eHedUK6sNXZMXuYlo1ZWdsibclrWX6sO5eJ+aipfVn7qzUoWrkOllU4WqZyWshDabzvAmlv5V4WH2/1DuAQLrn4scQ23SlM4Oc1VLqd+WwLLSZkBaqmKFZxvYoXvsJj8/jHq9YYItJpplUazKltIH7mlKCUcdr5Dg97gJOr3ov5Ze8xGHOEv6b10VBsKmN5Y5EDMPcJvbOJAJtfR6DTYAZatCmoVxsuZMSvXlfdlr9+b1sDTGniKG6PTbjBZaQ+ognBOWgRTeNdtTEffeYv6tGO6WwrPEC8doJA8nHFL2Fkruak9tTlc3GehvkuXn7joG2EzfmlofXV1TcKRwFWhKRUltZbCFtI5cqayqRzDvshWU1fNcuWcMkUX3hoN2rhuQyw8jMstEo8Q5L23Yjoz2faP53v0UOXGh+d/nuOkxX2+g8LOtHRwVAIoFz5dMuO0WSgYK61BmZ2xbYoiPkOfk8nl+6+t34Zt4tzSTIKPyLxLrZrxTeU5ntLA1ZneRmpRhFpt8HBjDQ6IIkNJAZYaa2Xq8XIpmaOwB0NaeqaPUyBYQHQV+FwVGRsHHVD6LVOicnJeaQodde/jZ4/oPqZ+3KP9p6Fd/2TPrcFgTL14QNEvfJKLCYPsYTICjbgJC3h10X0Dp7g/udBxMKrqTNH/fo5oXz7Eh6uZnAtwq1PH7RsLdHtz9ZUyNZ8jdI50Jv1GyiK8s6JYRUPwWOSg1yGsQEy3hLPr/hm1rt4grP5pO2CtB6PmyWDUphkyyHXmMrIWdIMUNOUTGuzLtgehAjXYPWWZZCL1mvXAPNaZkEOwlrRwnpzIS81VQ5epQFIoisl/mkB2uGn2mN4aBx4WBqa1jlJt0tWY05guRbFAPDuQmj6i2FLTaWglkIJD5VqfHYf/cfdk9OwH+RvTaf9s8ECVQe+BKsMxjXpnJ8wVPtQJFUpuLt5cXd78Sm9vb95d3l3e3gQtf9LtB6Nko43TaEeEi4oJq7gbY7qWArSXGVwnM6d+59XxECJISHLB2U8hsbVb3mQCYn0IiWvU6KiCXHhCIbi5zaY4R7Y89Yc1yAh53yl0jdRC5vovOvQNNXnV9cKt2geh3t3unFMMEzq4TgmjW6Ec97aKEeK3PrE9ltwRbUyloYdiJZGNVKuylNlBIHYvpo9SrMjMgpplLFZ5GRqsf9I5O4P9UFx6Tcf9+OwFx8i1Gw46r05OaaOKzGwOgrR2uoH6BPqO2oHxgSw1gXhGbzGatBR2B9BIUmBsfBCgtn/6nS7dfbi4o4+T5PryU8LjYpLcTS7e3iXvqPXDaOqfRnWfRIh9DSlHxqODiKLKVDPBHlurVwfePBopwDiCfGWsVYIZenX7JZmE8ZSjSA5LRhZ5Y9CRCPJwpaCdad3917eThLRaSVYOZj4EyZ2HnO1Es1HLuojD09FONdmdHL5mh/Xw6bgNLfLl8u7DjytPbZ+1//uKIQMocWakCy4iG+nyIGYF+SiZiDX/duYfaPiIfw33OKi/pTWhA8Aei2kZ0uCtKn8SIIb/0mh0EQZTgTGZV1rwDEWZAi9/IO1AotifeYNEQIUEDKedrpL3dwSqcYoOQrKwPSwfdHSMaOtpLHYbV1i1BFTfLjDxmCVWcWAa8x4PnJcldsVlGHQelMMm+2Q9HMMFTHxdZdgil6ZaLD3SCu7I4AJqIpHHvZiY+ZxajldmDJx6hZnuXr4Op9vnlAy6yZDwDvkJCgiMJwRptJ0x3t5OJknos4cNtFlAf7K0PSxh9f6G/ffwgoAGQ1bC+iRm6Nt23Q05VgG95VbIuUKRmu+3J4A9WryMVQuFhVwfntgL/GaA3sZ7Ta/1t44JYYTQeVcTPBPmCKdeErJmuAaaZipj5w4CKayjnGMeKhIbTerPaSVlyZwcHJ+Akry+h30oq3/LtKkqvNK4gRe8r3OMYIuMn/0HUEsDBBQAAAAIAAAAIVzUEqserwgAAMcRAAAQAAAAZXZhbF9jb25maWcueWFtbH1Y7VLbSBb9n6e4NanU2BVLlo1twGy2ikmcKapIyBKyO/PLaUttqxdZrahbOMyPvM6+xz7ZnntbsiHJkiJgbPX9POfc2zynD7qOvHK3tGqyjfaOelWhynib0X//M4kn/Zhuck0utbWuqdYqc6S/qtQX9+Rz7TStbVPTrb531JQZnvms71TRKG9sOf989uw5Kdoa50y54adoq1Xp6P0VFWZr/IBUmbEhylVdauco02vVFJ56swTnyiHH1qedbYqMVoXdycOjMeVUN2UbdAwv/1xcX7z9s/ePcX9OqS3Xpt6Kv1Jttdt7eRQcwaHnuNRGmdJ5ypRXw1rthk5tq0IvXbOS0G055HNLsbuJ79W2YJf/sjUOpQpFqJA4122O0BLimB19pd4kTugljWJJhZz2TdWnVzRK8E5Of8OrkbzaKK+p560t3BCODAJ56D2u7vtwCJeLPz5ESTKlnlNrjXbIM6jiOBnPolESjROkv0hoZ3xOV+8v/6St+rr0ZquXiKDxKIWvzXarM5rEU4r+jh8JmrD2SODgkJ6P4awXvB2hymVW6AGNTiTa3T7vPuXG0y/vrdcra2/pBo5s43+JOQblxYfDzw4xsCztO2R9xgHQxtxpB4/hfbsGqOptUwA5qt6gctw+48gh9oA8+Ikyi5RgAri4h78B7XKT5qQKRuk9QMA95I/FK47nJsu0tIG7t/ha6ZQ/Ti2yWdd2CyO/MgDvoskJyqRSVKuXNhyIR4CkC1U5nS0dR6gVfKEBtriDDe44zhYKpnRm/HJtCo1WjJLhaARn7YMOCWWaa2BqWiP6Qp7mBMbjY3JniBXprWFGVWY5mhwlI+rxI47LOZ5Mo/FsTG5PCE1pI8cVPkzIoSvVAP5MrGP6dvqCI+18D+hbEk/4HclzMpyeUNWsCpO2gI0iGOwoVuvK1lyel8NoHBTi8jcqrXEAgrOhokxB4yQ/lSE3w9ErxApqqTqCDVhnbItT6sHCnaqNKlNQprYrDXn5yFwTFqLJeHNbeVK1liAH0JQ0V+VGZw/xfwSRksCjwP/I5apCWUHnAxVGp6i/RIkvyTjQd8AF8ZCzrIV9a/NB6/E554TkemULbTfUX6tklCyD42VwvMSJyclwAKWBehVUowLbyXCjt1s1ifhEchQ9DlVOSAxQOcRWwhMe1GClLr0bttE8PgU5RqW65KG4rZy17Qv8/NX9QPY5nQzlPSjLMlVF4eZwNz5hcqyYPMz1PYhdYTY504sJDj0C3Y+nL+iuLd94HJ++GFD5iuN/fQEW3OkalOi36gqAckyejehSZ+IpYq+MSspMrYW8wjc8XGhmiy11FwH1QO98eTQ7ngFi3lYV9+igHhw359APzgKKp+IBvkKlQH8gmBlpmWDG5QzrzAK5XlBqAa/vSsKGWTChXDjD2mYQj5LRYFpp4MwsvtXiicsN4RBkQiOc1iGkzKYCo9SIijokjJ8PtXt0whi/06VwQDXM/t5evccD+kvXln7/8OnsR1SMUE2BcHQwEYmJgI/XditPi4hwozFledxpoCwwgVWJdiBoSyrW2UeyP2VlpOjHqcEqbZjYFboX+aDzxFPIQ6/AX6hhOy+mP50notnQ4sb3B9IMxZ6I9wHX1JoDbxE+mnWFoyOuOyMUSHO6xTwehghxZ1AGLxo+55f/DnJ+/vrm0/mluOOIOfBvr+g4PqV8EFzyVHpFJ/FshGHTawPIzlih8OoOUp8hxikOvqLTGLKaD2gmv2Fwn7IdmJNfx/H0mG1UjOT9YoJVJgvrTb/12C0YigFsVrVMLYv2CgMxBVJtRAOhk9/gZEo5hNFK8I7F/9uY32vlmRcuuyv3h0Q41b0Lvvajhnvbzs+d7ihQangGYIs1QGRSJtgKWGKaPZoODwdDvwXFm8XlxW+L6/ObBfaKy8XbGzq/kRB/QNGcldC1/FEZdsYOMbVxt2HpU/v3ZLl0AfnMpZDHI/QA3dcHi97uuMToYzQNuPzS6KaFM7ch4GhCwz3p1pZ1IWqqAdgB5gWpD56+NNYr0M0472J6EwoVGMLeOmkQhmMoPKTIXkQgRBxHacsIwppJL8tHCwWAlmHy0dE4qBjWKa42zxuZvq18Qir7MmDhmZWy7SpCytkoZO1W6+qMWOXqfT1Ywt5cLT4Gcd1DoNvg5mK6VWN8E2VNbY4Jf5D/hMPBAhk/e7AkP6OfDBUmEFGraNh39/p1NBc8yFoJGbhefLy5ur54/3soBGO2Y5uMhF7Y2GpVPqP/++95x1RdZpG3kWbs4P+eqoCG/tIYNrrSqWqcDnMI9brlArmdkAQofNLLS4R8RG7om7o8o9wW0kROJ+xaWMnqOwZELPmzGBayB2l6d/7HxbtP74I8qRTLwJOeZNuXOYVaHpZpEZSErw1cm3BtaK8MqLxgot3Z24H4pBPesZNWmxwvfxyceOb15tE2vcFc9HwNBOTDMJ+GJkLf2MaTfnqidrqubd2tKDLjRFvYpwhEGBCdyGANFMzHHboezOLZpLMc0DU6oGsyZ5YhrtkEw05Y16oaUw5gxzWr4xjYNQS1lBdC8dUAk+/m4t0CR5gYx6fx+EW4RwQ69A9brQlyFZW6wS2g4PUacxflamQqzTue7m8pRvbuiMduTG9rzbcseeRANq9MwemiYKgNZCoNw7rPKQES2CUBMN+4Ftt8rVtWynPVsAyvYVQ+aRNOsQL6rnoALAp3OnvUl7Z6czqWVp7OeEnl1Vrk/jvx4szZCsYDJm0o50ojvZAEE6cV6yVGsy0z+DsCNMTTw5s334M5s6Y02AscJlWqZe0S7eLinl9+vOJqhRJ9yAH8aAwIhkeqe1m+nwIcFFhahZnGSvjTPx7wsOUbCQSNb+XQntEUZAp/y1Ab1D1yJtNP+uE0ZLFtpxRiBw1739WBN3m+90h+fEnizwOWuECi4J14nz3pL+w2cv3Z1VyfUAve3T53MZwmn1nsV3IB4NFa3u/Uffzsf1BLAwQUAAAACAAAACFce5aE2bIRAADcKwAAFAAAAHByb21wdHMvc3dlX2NvZGVyLm1k3VrtchvJdf3Pp+hyqkxwDYwordZxyChV0orKsrwrqUQqiZxKAYOZBtDmYGY03UMQLv/wQ/gJ8yQ5596eD5BSxP1rVUkCMD3d9+Pc7/5UtSZtrPE7O8+q3DZTkxpvS1c15v0+bKrS2HLtSmsbs3J3rlybd28vjPO+tcaVJmysaWxdeReqZm/SYJ7squbG12lmE/OTy3OLRdYHb1bYMmycjy+vmzS3Zl+1janTkG3OjAv45L31piqLvXErLK+8ja/zkUnL3Ng75wMJ0d9vrK3lIX5KzCfwQwJMtVoVINu0JZgCT5u0yc0uLYpZVlTZjewUqgpf8ZtZtvnahsT8J19dpT5MjQ/p3thbBwYyO1um3uZTecu3yy1ITU1ZlTO7rcNeGTBLCxatiET3g+hynxwd/ZO57k/Knc9cLaRN7K2FzPjzFJuBsczWwVWlPzmamYu7NAsQA8VNQmWdqcFMaJsyMa/0NAdSd40LlrLfVj6YsKuM31RNoB4DqffnpuRZ0BS4CkqiagE6sWlTODys2lC3kMEfKdC03JuNaq+xqa9KEXgFrld2N2w85ValAaUJSL7GtiLTMt0CUkUVzKYqcg8efv4EpeI46tRCp1zhz0zTloDddgu5TnlQPl+5wk6NzV2IH4W5+BkynZODFgerGuYi+sS8Ff5AP+G7sZBUvy1IxsvU0YbskrMKRECO9i50GO4pJh8vm3W7BYteTIOPiDpRxwHJJqQ3QGv8lgwMxAf8KMfWRQtQi27TgshqwlwgIHguc/0ycZDq2jb+JBkEcG+rqYFAIYMG+gBo7C5+TkZyevBKVlFdUO0gvxGSVYTyElGYdsx3QnXlLb5GmXVPVYGyi0JLMXhj9y9u00J8g4fpQPiELYV6ucKXuoFFVTgeSKwBdAuThAChNtgrERwEQjfAVOG2xHZegahAH2Pj4x6PZ8bSEEV9VOVgJW4LlTkgHfYjjMLggilAvBXJwgeVALPfYjGRX1o101dqtoXFf83RFfVk1Ehpc2bnCCFoatmk2Q0XQm1TcWyAx7Yu7Jn573oDV/Hi4vXltdDiXzz95yfPf9BjXzyNMg82f1FW/5OYH6sWwhQvWO1KfeN8rCf4y1VjFSlrdwuZCxJN2W6XgMrUtJ5OwKRL2G8k9ukz3Uhe2sFARUNt6dsGTvlC1vygS2Aeha86D2K+nwkSr65fXl+Yt+/wD+yjLlJaCUU8Iag8JN1mG/rDrG0aomGzr4kP7/xUdeGDrU/Oe98iqlEHo0aVN1Vdw6xWTbVV9gWkd0F9rMQJ6J1uChxA86D9luwTSm8uP1xdz0TEry9evv758u2FvgY5VOCuAfJ8lIWoA7zWcM9FtTtjWJHzxjqUtXSaO99JLlCKUOyWTna3oVlF5Hgw5wCy06luJMfcA+AvH6+uceBgxol5rUD2EEm2UWfH/QHspsrbzAqw5LTg4ItWrvGw2GtaJREurt23vnaZGJCQjfMR2G6huYL77bE7vTEkkW3Sci3AAM5FgMo0AsAXlGZ8uvdcDBG0RY5lU4qkjMISjXXw+RS/kxl6PoA5xgYXzky6ClQ3DxQORAKyh8Rp+hJIb53KOyS0sTMu4deyamiQe9VlClBW+FU2yNKS4mwsgQpflkisUU+gYGlaaoipAcIKVo8Dc1rsyJ/PIF1v/mKbahpVmsYVQjB22dqGPqNLbNx6E8hs2iUmEEHdOKYaWFWB1xKBqCIh2cZFYtUHYz/xRF52kjfgN9x6TQ3ADeYuQ8bEgEiyxINfrgaFwhGD59zlDNcaKnTDpSYMQKdGSdl+1ZYZg0uMVZF8jfAM2nk+YAjW3Hq3BDXrljlRJaZQihC6d2A+WeOW0Sv+xGWUr4ftvW/sShKqyNKi1ixxlpnjJEmOF8SxeJFGbFFciygbjDYi7JF8eqYD9qcV2TzG6ynhuK7E2ZgnYVtLUjg1ty4dh+DerugGRyGQPnl8HkAxehoBIb/4mCIOeWvEXwnGSHhhV5ImDAsARRxvmZM2jFyjLLZnSDxKBmAEYemA94PNEOJsYYPYqgSYcYIRU8oYNAQlF12WCJjD95Q5FuQ4PLtJYfDiT3/zx5LBxGOHZXVn1mntf6Puz6hbvU+Q2DrU2K43JKOQOHNfLOZ///b3GOx7iumNNUvYVrlb7YkmZVqzc8k9VvyY1FAesMKPrnT8DMf3Z4vMMVRbJMDehrZOstWae4TqjqvOBJNSKUjyaoOY0zYxb9ydPCscwMskGrULiYloQPYRJF1WsWgkjKlEYDiPNquYR+BpShvO1VSj1CCproaQLI1uXgFBlNLKfO9243E2P1APnOdCOTazz7QLxUlgGhJU8KvUFYJw0UBWYY8swmDxC8JCYd9W4Q1yhPyiaarmzLytKGn6OjX1Y1eSxLkvoeRNFWh/jVkc564J+7n93CK8Hy+m0cFta5YFlluR/LaEtxOAggKhDSWemaTmRvAD3KgrkQAhUjn2oteZFGg5crKS5ZGzTFmvkKX8f8iDjCTmrZjHUI4tbVozvtTfDFCCAcivOxQPiiMYACw1Mf8OJPDNxmJNF+FlAegUfYpUNZ/QCKipiidXst9MVnR6YX78JFRP+GV+B4wupBIrgSYAksqAJk/Ex+MsAlySUOYjLBiAzLJFWukCCgMpIcQbi/uKZqZeUjZ0sYCE96ecIwmibIZ85m//cpqYl10127015NKa3cvxkoIofOhmCfYY3xTEoCWlrVOlifhtZFOZhouYauFoeI/dYUmjSZrZtiANEXdJ7BHX711tY/IiZzTiLDbU6Kw0z08JO/2W4dvp6bmp+YawKMrqXgmgma/8/rQrMIayKeooZur86azz1VL+f61yyigCCTKxBn767FSiJpNNcqBC2W+XSNDonWnCUumVwc2QU8Fa1bNp/EtjGQFwB4ePEkRZN0ZXX6Q443tNFROtbDr5RYveOXr2DoM4PK3h7ojQCfwkIigTsM8t8Av/3/+gyBm+by2ccn5Cr8IkiHB4qi5+jbwRj9NSdLzVBsrouG7rrnqi01N3LeW9wvTObOFkkXNpdYE6ATg6o5sorBR5kgS1hf1SatAjvLaZS5n3eg27TJJjOZSYj7VkMEw4qJ9+61T6HKN9+cTRTkfVFHffl1kMx8Azv9zC3NIyQCi/xNRNkXJMeOx55ls691XK/EqoBM3Ldg2zg6PXH6gsre+lahpcELKdhx7MTDTM9lIjBuAdVtMImpUlIWQ9hh0XTo6udxVdZ5PPmCXsR35zCEoM7UgkaP3IPZ1aveh37HQPwvNhww1Mk1rthEX3rgGSoKQ5x04aQY/i4LAJo5XklwLNwkvd2EUkklWJY5Y+kHquaR/pmVpG4rs8pBMRnVDKPgVMqJ4V4IQtsMZm0i7UT6DwQa4xSXvMKLmjtl+/UTysU0Ns9cFOFyKQJ4vhqBP1s+NmknSuJJJ0eQmKCmlPam0Dbxaky+CkNEhp2mGzaguxGhTnTSdKerCiS0WgkroF5Zl5+f6yN5LQLtVxj9wZ0mu1RifxKwZgdoiGhqj4YXX28MY1cyDYpeiGKhU3NW7jaISRjHeu7oDhg4oS9aB6IALUTMmVvYP5BpbySij0KI0gODyIs0J6zVRPTDc8zEZZvYWGRU3jo5KIqC8ASqj9M5mBATEjHlxlzPBku7aEDQK6i2/nOQcJ/eLe4yfzOXQU5nOGdFV918A7WiwWR1nBcHIVV58dGfyZz9n/8/O5eWEmx3NpYx1PT+RZblem23Kipi/PX7yFOE/0ff7ho0RfxS7y/+h9OXWO5A90b7FRxv4LP492gKtmqwmPRi/az/2x0oZ7uP7g4Be6arQBRPONDbQ3wU30NF022oKNirjJw7f5cDKiYfziJvWbr77Ih4cvHvGtTo+TB1KO710dLDgRpX4ZOIeZ8CFqxs8eB5n5a77yCrHubMRi1XxLvPN3zcGCsWAf97Y8moqOHi3daikF1uGqo44VbDqwc3L2NaCnU7O8D/EU6E4Pf1rip+Wvx2zaw5URptvpIYIfCb/0xPzOHJu/4u/vRj8ve64v/WuXhRHjzLey8HX2v0MuCjP97rubHT+NDud7yTdWP04gCNBStXpJIMrMdtoekXaP5TeArL0vBy6PlksZytcDnV/690hGkK79Iwph3iUhLyLwJyf3BYRwqbsm8MKTm2n/jsjrVgIjfr1lPBbg0Dv7yclIgO9K+271KMP5TnyTv287+msXHn6dm0dyo/QP5A60xuMeazTHHS+HphJ3oRmdHHds/5T6D1zxGLa7lEJOv8/8wUPI4OD7rxKFUDxg/eHuj5ZDx9yhHA7ZiOJgIIjJZYukzEus6HokTMuR1GwdU27Jxf2DxCTpAtcC+RXqcLa+ORRka9NXXaYe642D+LQQbaf3ftSKgpkU0tE4zOnevnf0YugMIAU1fT2DQLhjfjm0FJj9xcw/QYnczUs784rxVXJiyXxrNvG79NyWt66pSqkfpbTT1PKg8Ai76qyrTeN4MR1amJKcTuP4Z0iNR0MpZo9b1PzYVWo2NjcqUGwmMu7Rdp/kxhl4P+/Wat6cyjRI9JVyinVy9DQxP7/7kbtPugbCH7S4l9I7NExq49hNxp6xRIu161nff2e/s5DMfsreMNYGmdH1k30T9jUfSgtuhq28Nm3TdRy5XkmH5WEjuBXJr9kTZ+U/i8PIYTLEKnyFogMHbrXxHq9vSC0w7Ro5+trQtpFBSjmeXLM/1E8eZHVqggxCpNXSsKaQ8rIbm+csdDS3Z194jfc8h6IFu366HRRB/cfyDxXrebwlsdY2awAJXKA3JFASwUEfPUvMh4v3H969/vjjSDHP+rkl57o3rq5jcSLtWBxp2S95R0rHXa2So/ssH5eYv/2tuTe70ApN+wRyVQCabLTljNpnk3J03Xy5F8JmV/6gx0X5vq6I2X7CKvM+oyNBCHHgmqOOOOmOA0EawbqCXTapyB9WKTNPUcowzhuN8Waz/nVBm/PSf3fNVvotXZEHe23gPxBpg02Ovk90217ET5/34L/qr4uMTnGqrO4SiJmMdIln/8XxSMp5zKdzbUlRQ3+Swo9DRmz70Y9Go3Gs3l9rAMm1G25cDMamZfszyuE5vKf73OpojJdB5L6LNPnFDdFxxUmVt9Ie1WH/6FSd84qlCDal5YhPopZhGZ26YgDiGRG5S1VdK2kvaj9d5lBC1/TAovCmmI1J15yk69iTHbloXkI9Mx0Z/3YcOM51ywzPhus7Mrvo+7vyEB5aZq7S/ibd0/vWHNsASkJFjYl/F1wkR88T8x8XHy7ffBoB4LQHwAfbdeMFWJ27aMuxeZ2N+te9TW1NP/vo2vyjhj6pmyDLkPPuTyFiH3PypRb62bhZ4OP0k3FL+sdi6ryfIVdI5CSkTHD7jUzd4CaGpgM7fidfs1x2p6Wvq9TyJQ7qMvbhe0DGXjm2l/g4pl0MgBcSzqVjqG6QNqHKZ6Mj6u2Z7M6rEFiX7TPx1j5UtTZlK0rl6uOrXzrDV2cEooAI73SoJDPj5OiHJK5MzI8E+/i6lHS7DjtFB7dk4tzeam9bhh1ihAAUdtUrb9KH7HpP8cLKxmY3w52Ixo4VZHsVbcYXAcUKfLyR4PFPZqXnLRkQO+kNm3by03imQX31sobBER3aja/YoldKpK8W5yxlN00qdf6scNAxelaREmyxkaHqq+HqnnbgVFLT/raIENPNkthi7MdHU7GGBUOzkEAzgB/+V40lWPBvC6qQF17MIEttqY3u9vQ3HGoZGZ9r57ZXiNDjRd8Zrwz57pIELae/PCPubMW4u5N7HAWvT/LiRrxzJFcjRczxEiPv5+gVqeiaBAarthEhI4ltsGNMTV72EBmLqAMTCC96h4b8Ti8S+RYOotETtvGCWbd7f1tGbxm8qYCDfMTo0QfkEk0Oe2CKoBd5pgq/0f0ovbSzsju9dlIicxs2JkxGj35/imjApJu53xb8yiU6rOn7qXL3QixXJwN/OOWNy0zuv62GZEavd0YzRaFQyA2OMwrGeTXR/t4Lm/wMvUSJRHuCZDpkqIf3GF/KnQLOU+gOIvr1VhA5gPhk4fm3r7gkR/8HUEsBAhQAFAAAAAgAAAAhXP4YlhkUAwAAMgUAAAoAAAAAAAAAAAAAAKQBAAAAAGFnZW50LnlhbWxQSwECFAAUAAAACAAAACFcAZqhGjYGAAAuDQAAFQAAAAAAAAAAAAAApAE8AwAAY29uZmlncy9zYW1wbGluZy55YW1sUEsBAhQAFAAAAAgAAAAhXNQSqx6vCAAAxxEAABAAAAAAAAAAAAAAAKQBpQkAAGV2YWxfY29uZmlnLnlhbWxQSwECFAAUAAAACAAAACFce5aE2bIRAADcKwAAFAAAAAAAAAAAAAAApAGCEgAAcHJvbXB0cy9zd2VfY29kZXIubWRQSwUGAAAAAAQABAD7AAAAZiQAAAAA"               # filled in by tools/build_notebook.py --embed

# Compaction settings: Q7 ANSWERED [V] from HARNESS_README.md section 7.2 (interval 15, overlap 2,
# token_threshold 14,336, event_retention_size 5). The 'interval 5' reading confused retention with interval.
COMPACTION_INTERVAL = 15   # [V] HARNESS_README §7.2: interval 15, overlap 2, threshold 14,336, retention 5
COMPACTION_TOKEN_THRESHOLD = 14336

In [ ]:
# ---------------------------------------------------------------- environment + wheelhouse (hosts' recipe)
# Adapted from the organizers' Getting-Started notebook as mirrored by community repos.  # VERIFY
import glob, importlib, json, os, shutil, subprocess, sys, time, zipfile, io, base64
from pathlib import Path

os.environ.update({
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'TRANSFORMERS_NO_TF': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1', 'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
    'VLLM_NO_USAGE_STATS': '1', 'OTEL_SDK_DISABLED': 'true', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})
WHEELHOUSE_DIR = next(Path(p) for p in ['/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse',
                                        '/kaggle/input/gemma-4-developer-agent-wheelhouse'] if Path(p).exists())
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp_whl / name).exists():
        os.symlink(w, tmp_whl / name)
# The wheelhouse ships cp312 binary wheels (orchestrator env = Python 3.12), but the grading sandbox and
# Kaggle's CPU image are Python 3.13 (docker/Dockerfile.sandbox is `python:3.13-slim`). A single bulk
# `pip install ... check=True` aborts the WHOLE install when one wheel is incompatible, which is what killed
# the first EXP-000 push (apache_tvm_ffi cp312 on py3.13). Filter by interpreter tag, then assert that the
# packages we actually need for this run arrived.
# Import names, not distribution names: the `google_adk` wheel installs the `google.adk`
# namespace package, so importing 'google_adk' always fails. [V] google_adk-1.36.1.dist-info
REQUIRED_PKGS = ('swegemma', 'adk_submission', 'adk_eval_core', 'google.adk')

def _wheel_is_compatible(path: Path) -> bool:
    """True when the wheel's tags intersect this interpreter's supported tags."""
    try:
        from packaging.tags import sys_tags
        from packaging.utils import parse_wheel_filename
    except Exception:  # noqa: BLE001 - fall back to accepting everything
        return True
    try:
        _, _, _, tags = parse_wheel_filename(path.name)
    except Exception:  # noqa: BLE001 - odd filename: let pip decide
        return True
    supported = set(sys_tags())
    return any(t in supported for t in tags)

all_wheels = sorted(tmp_whl.glob('*.whl'))
usable = [w for w in all_wheels if _wheel_is_compatible(w)]
skipped = [w.name for w in all_wheels if w not in usable]
T_INSTALL = time.time()
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall',
                       *(str(w) for w in usable)], capture_output=True, text=True)
if proc.returncode != 0:
    print('bulk install failed; retrying wheel-by-wheel to isolate the bad ones')
    failed = []
    for w in usable:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', str(w)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            failed.append(w.name)
    skipped += failed
importlib.invalidate_caches()
print(f'wheelhouse: installed {len(all_wheels) - len(skipped)}/{len(all_wheels)} wheels in {time.time() - T_INSTALL:.0f}s '
      f'(python {sys.version_info.major}.{sys.version_info.minor})')
if skipped:
    print(f'  incompatible/failed ({len(skipped)}):', ', '.join(sorted(skipped)[:12]))
missing = []
for _pkg in REQUIRED_PKGS:
    try:
        importlib.import_module(_pkg)
    except Exception as _exc:  # noqa: BLE001
        missing.append(f'{_pkg} ({_exc.__class__.__name__}: {_exc})')
assert not missing, 'required packages unavailable: ' + '; '.join(missing)
print('  required packages present:', ', '.join(REQUIRED_PKGS))

DATA_DIR = next(Path(p) for p in ['/kaggle/input/competitions/gemma-4-developer-agent',
                                  '/kaggle/input/gemma-4-developer-agent'] if Path(p).exists())
RUN_DIR = Path('/kaggle/working/runs') / EXP_ID
for sub in ('traces', 'patches', 'logs', 'phase0'):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print('data:', DATA_DIR, ' run dir:', RUN_DIR)

In [ ]:
# ---------------------------------------------------------------- Phase 0 checks (plan §3.4)
# Answers Q1/Q2/Q3/Q7/Q9 by reading the installed harness source; saves HARNESS_README.md and
# sample_submission/ to the run dir so they can be downloaded and read offline.
if RUN_PHASE0_CHECKS:
    import importlib.metadata as md, re
    report = {'python': sys.version, 'packages': {}}
    for pkg in ('swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'vllm', 'litellm'):
        try:
            report['packages'][pkg] = md.version(pkg)
        except md.PackageNotFoundError:
            report['packages'][pkg] = None
    for name in ('HARNESS_README.md',):
        src = DATA_DIR / name
        if src.exists():
            shutil.copy(src, RUN_DIR / 'phase0' / name)
    ss = DATA_DIR / 'sample_submission'
    if ss.exists():
        shutil.copytree(ss, RUN_DIR / 'phase0' / 'sample_submission', dirs_exist_ok=True)
        print('--- sample_submission files (Q2) ---')
        for p in sorted(ss.rglob('*')):
            if p.is_file() and p.suffix in ('.yaml', '.yml', '.md', '.txt', '.json'):
                print(f'===== {p.relative_to(ss)}'); print(p.read_text()[:4000])

    def grep_pkg(pkg: str, patterns: list[str]) -> dict:
        hits = {}
        try:
            root = Path(importlib.import_module(pkg).__file__).parent
        except Exception as exc:  # noqa: BLE001
            return {'error': str(exc)}
        for pat in patterns:
            rx = re.compile(pat)
            for f in root.rglob('*.py'):
                for i, line in enumerate(f.read_text(errors='ignore').splitlines(), 1):
                    if rx.search(line):
                        hits.setdefault(pat, []).append(f'{f.relative_to(root)}:{i}: {line.strip()[:160]}')
        return hits

    report['Q1_tool_registry'] = grep_pkg('swegemma', [r'register\(', r'def (run_command|read_file|edit_file|write_file|get_status|submit_patch|get_code_neighbors|search_similar_code|get_code_subgraph)\b'])
    report['Q1_skill_tools'] = grep_pkg('adk_submission', [r'run_skill_script|load_skill_resource|SkillToolset'])
    report['Q2_eval_config_keys'] = grep_pkg('swegemma', [r'max_time_minutes|max_tool_calls|max_turns|timeout_seconds', r'load_submission_eval_config'])
    report['Q2_agent_tool_schema'] = grep_pkg('adk_submission', [r'agent_tool|config_path|skip_summarization'])
    report['Q3_thinking_forwarding'] = grep_pkg('adk_submission', [r'thinking_budget|thinking_token_budget|enable_thinking|include_thoughts'])
    report['Q7_compaction'] = grep_pkg('swegemma', [r'compaction_interval|token_threshold|EventsCompactionConfig'])
    report['Q9_extensions_limits'] = grep_pkg('swegemma', [r'ALLOWED_SUBMISSION_EXTENSIONS|MAX_SUBMISSION_SIZE_BYTES|allowed_file_extensions|max_total_size_bytes'])
    report['Q2_grading_timeout'] = grep_pkg('swegemma', [r'timeout', ]) if False else grep_pkg('swegemma', [r'verification.*timeout|timeout.*verif|command_timeout_seconds'])
    (RUN_DIR / 'phase0' / 'phase0_report.json').write_text(json.dumps(report, indent=1))
    print(json.dumps({k: (v if k == 'packages' else {kk: len(vv) for kk, vv in v.items()} if isinstance(v, dict) else v)
                      for k, v in report.items()}, indent=1))
    print('full grep hits saved to', RUN_DIR / 'phase0' / 'phase0_report.json')

In [ ]:
# ---------------------------------------------------------------- tasks + candidates
from swegemma.models import load_tasks  # [V] swegemma/models/__init__.py exports load_tasks(path)->list[Task]
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
from collections import Counter
print('repo counts:', Counter(t.repo for t in all_tasks))
if TASK_IDS:
    want = set(TASK_IDS)
    candidates = [t for t in all_tasks if t.instance_id in want]
else:
    pool = [t for t in all_tasks if (not REPO_FILTER or t.repo == REPO_FILTER) and t.instance_id not in set(EXCLUDE_IDS)]
    candidates = sorted(pool, key=lambda t: t.instance_id)[:N_TASKS]
print(f'{len(all_tasks)} tasks; candidates ({len(candidates)}):', [t.instance_id for t in candidates])

In [ ]:
# ---------------------------------------------------------------- unpack our submission
AGENT_DIR = Path('/kaggle/working/agent')
shutil.rmtree(AGENT_DIR, ignore_errors=True); AGENT_DIR.mkdir(parents=True)
if SUBMISSION_B64:
    zipfile.ZipFile(io.BytesIO(base64.b64decode(SUBMISSION_B64))).extractall(AGENT_DIR)
elif Path(SUBMISSION_ZIP).exists():
    zipfile.ZipFile(SUBMISSION_ZIP).extractall(AGENT_DIR)
elif not RUN_AGENT and (DATA_DIR / 'sample_submission').is_dir():
    # A gold-control run never calls a model, so it needs no submission of ours. Fall back to the
    # official sample_submission: it is a real, schema-valid submission dir, so the model/adapter/limits
    # validation below still runs for real instead of being skipped.
    shutil.copytree(DATA_DIR / 'sample_submission', AGENT_DIR, dirs_exist_ok=True)
    print('no submission supplied; using official sample_submission (RUN_AGENT=False)')
else:
    raise SystemExit('no submission: set SUBMISSION_ZIP to your uploaded dataset file or embed with tools/build_notebook.py')
print('agent files:', sorted(str(p.relative_to(AGENT_DIR)) for p in AGENT_DIR.rglob('*') if p.is_file()))
import yaml
EVAL = (yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text()) or {}).get('evaluation', {}) \
    if (AGENT_DIR / 'eval_config.yaml').exists() else {}
if not RUN_AGENT:
    # The official sample eval_config is deliberately tiny (1 min / 10 tool calls) to demo the schema.
    # Gold control runs no agent loop, so those caps only risk failing a sound task for budget reasons
    # and corrupting the Q5 answer. Give the grader room; the agent budget is irrelevant here.
    EVAL = {**EVAL, 'timeout_seconds': 300, 'max_time_minutes': 60.0, 'max_tool_calls': 100}
print('eval_config:', EVAL)

In [ ]:
# ---------------------------------------------------------------- vLLM server (hosts' recipe)  # VERIFY
# CPU-safe imports only: a gold-control run (RUN_AGENT=False) never loads vLLM/litellm/torch, and on
# Kaggle's Python 3.13 CPU image those cp312 wheels are not installed at all.
from adk_submission import ModelRegistry, discover_adapters, discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, EvalConfig, build_submission_limits

TARGET = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_DIR = '/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2'
# [V] adk_submission.discovery.discover_declared_models(submission_dir) -> set[str]; walks the root
# agent, sub_agents and agent_tools. There is no `validate_single_declared_model` anywhere in the
# wheelhouse -- that was an invented name. Exactly one model may be declared, so assert the set size.
declared = discover_declared_models(str(AGENT_DIR))
assert len(declared) == 1, f'submission must declare exactly one model, got {sorted(declared)}'
assert next(iter(declared)).endswith(TARGET), declared
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
# EXP-000 gold control needs no model: it applies each task's reference patch and runs test_patch.
# Gate the server so a gold-control-only run costs ZERO GPU quota (plan §5.7) and can use a CPU notebook.
NEED_SERVER = RUN_AGENT or MEASURE_DECODE_SPEED
server = None
SERVER_START_S = None
target_id = None
models = ModelRegistry()

if NEED_SERVER:
    import litellm, torch
    from adk_submission import VllmConfig, VllmServer
    from google.adk.models.lite_llm import LiteLlm
    try:
        from adk_submission.resolvers.generation import install_litellm_reasoning_patch
        install_litellm_reasoning_patch()
    except Exception as exc:  # noqa: BLE001
        print('reasoning patch not installed:', exc)
    litellm.drop_params = True
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    assert gpu_count >= 1, 'no GPU: select the L4 x4 accelerator'
    tp = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
    t0 = time.time()
    server = VllmServer(VllmConfig(
        model=MODEL_DIR, port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
        default_chat_template_kwargs={'enable_thinking': True}, max_model_len=32768,
        dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto', gpu_memory_utilization=0.90,
        enable_auto_tool_choice=True, enable_lora=True, max_loras=8, max_lora_rank=128,
        tensor_parallel_size=tp, startup_timeout=60 * 20), adapter_manifest=adapters)
    server.start()
    SERVER_START_S = round(time.time() - t0, 1)
    print(f'vLLM up in {SERVER_START_S}s (tp={tp}, gpus={gpu_count}) at {server.base_url}')
    target_id = getattr(server, 'effective_model_path', MODEL_DIR)
    for alias in {next(iter(declared)), TARGET}:
        models.register(alias, LiteLlm(model=f'openai/{target_id}', api_base=server.base_url, api_key='EMPTY'))
else:
    print('vLLM server SKIPPED (RUN_AGENT and MEASURE_DECODE_SPEED are both False) - no GPU required.')

In [ ]:
# ---------------------------------------------------------------- Q10: decode speed probe
DECODE = None
if MEASURE_DECODE_SPEED:
    import urllib.request
    body = json.dumps({'model': target_id, 'messages': [{'role': 'user', 'content': 'Count from 1 to 400 separated by spaces.'}],
                       'max_tokens': 400, 'temperature': 0.0, 'chat_template_kwargs': {'enable_thinking': False}}).encode()
    req = urllib.request.Request(server.base_url.rstrip('/') + '/chat/completions', data=body,
                                 headers={'Content-Type': 'application/json'}, method='POST')
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=600) as resp:
        out = json.loads(resp.read())
    dt = time.time() - t0
    ct = out.get('usage', {}).get('completion_tokens', 0)
    DECODE = {'completion_tokens': ct, 'seconds': round(dt, 2), 'tok_s': round(ct / dt, 1) if dt else None}
    print('decode probe:', DECODE)

In [ ]:
# ---------------------------------------------------------------- evaluator helpers
import asyncio, concurrent.futures
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig  # [V] google/adk/apps/_configs.py:50
from swegemma.evaluate import Evaluator

def run_sync(fn):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
            return ex.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

def make_config(results_dir: Path) -> EvalConfig:
    limits, gen = build_submission_limits()
    return EvalConfig(
        tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=results_dir,
        submission_dir=AGENT_DIR, models=models, sandbox='subprocess',
        timeout_seconds=int(EVAL.get('timeout_seconds', 300)),
        max_time_minutes=float(EVAL.get('max_time_minutes', 60.0)),
        max_tool_calls=int(EVAL.get('max_tool_calls', 100)),
        max_turns=int(EVAL['max_turns']) if EVAL.get('max_turns') is not None else None,
        limits=limits, generation_constraints=gen, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=COMPACTION_INTERVAL, overlap_size=2,
                                                        token_threshold=COMPACTION_TOKEN_THRESHOLD, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
        wheels_dir=DATA_DIR / 'wheels', verbose=False)

def record(r, task, t_start: float, t_end: float) -> dict:
    patch = r.agent_patch or ''
    dur = float(r.duration_seconds or 0)
    wall = t_end - t_start
    return {'instance_id': task.instance_id, 'repo': task.repo, 'resolved': bool(r.resolved),
            'status': getattr(r, 'status', None), 'test_exit_code': r.test_exit_code,
            'duration_s': round(dur, 1), 'task_wall_s': round(wall, 1),
            'setup_s': round(max(0.0, wall - dur), 1),   # everything outside the agent loop (setup + grading)  VERIFY
            'tool_calls': getattr(r, 'tool_calls', None), 'llm_calls': getattr(r, 'total_llm_calls', None),
            'total_tokens': getattr(r, 'total_tokens', None), 'error': getattr(r, 'error', None) or getattr(r, 'error_message', None),
            'patch_chars': len(patch), 'patch': patch, 'trace': str(getattr(r, 'trace_path', '') or getattr(r, 'trace_json_path', '') or ''),
            'test_tail': (r.test_output or '')[-1500:], 'max_time_minutes': EVAL.get('max_time_minutes'),
            'max_tool_calls': EVAL.get('max_tool_calls')}

def append_row(path: Path, row: dict) -> None:
    with path.open('a', encoding='utf-8') as fh:
        fh.write(json.dumps(row) + '\n')

In [ ]:
# ---------------------------------------------------------------- EXP-000 gold control (Q5)
# `sound` must exist even when the gold control is skipped, or the agent cell below raises NameError.
sound = list(candidates)
if RUN_GOLD_CONTROL:
    from adk_eval_core.tracing import SessionTrace  # [V] tracing/__init__.py re-exports it; no-arg ctor

    class GoldEvaluator(Evaluator):
        """Phase 1 returns the reference patch; no model call, nothing is shown to any agent."""
        async def _run_agent_sandbox(self, task, *args, **kwargs):  # [V] evaluate.py:194 -> tuple[str, str|None, SessionTrace]
            return task.patch, None, SessionTrace()

    gold = GoldEvaluator(make_config(RUN_DIR / 'gold_control'))
    gold_rows = []
    for i, t in enumerate(candidates, 1):
        t0 = time.time()
        r = run_sync(lambda: gold.evaluate_task(task=t, task_index=i, total_tasks=len(candidates)))
        row = {'instance_id': t.instance_id, 'repo': t.repo, 'mode': 'gold', 'resolved': bool(r.resolved),
               'exit': r.test_exit_code, 'status': getattr(r, 'status', None), 'seconds': round(time.time() - t0, 1), 'error': (getattr(r, 'error', '') or '')[:300],
               'tail': (r.test_output or '')[-600:]}
        gold_rows.append(row)
        append_row(RUN_DIR / 'gold_control.jsonl', row)
        print('gold', t.instance_id, 'resolved=', row['resolved'], f"{row['seconds']}s")
    failures = [x['instance_id'] for x in gold_rows if not x['resolved']]
    (RUN_DIR / 'gold_failures.txt').write_text('# gold-control failures (exclude from cohorts)\n' + '\n'.join(failures) + '\n')
    sound = [t for t, x in zip(candidates, gold_rows) if x['resolved']]
    print(f'gold control: {len(sound)}/{len(candidates)} sound; failures: {failures}')

In [ ]:
# ---------------------------------------------------------------- baseline agent eval
summary = {'exp_id': EXP_ID, 'server_start_s': SERVER_START_S, 'decode_probe': DECODE, 'eval_config': EVAL,
           'compaction': {'interval': COMPACTION_INTERVAL, 'token_threshold': COMPACTION_TOKEN_THRESHOLD},
           'tasks': [], 'resolved': 0}
if RUN_AGENT:
    ev = Evaluator(make_config(RUN_DIR / 'official'))
    for i, t in enumerate(sound, 1):
        t0 = time.time()
        r = run_sync(lambda: ev.evaluate_task(task=t, task_index=i, total_tasks=len(sound)))
        row = record(r, t, t0, time.time())
        append_row(RUN_DIR / 'task_results.jsonl', row)
        (RUN_DIR / 'patches' / f'{t.instance_id}.patch').write_text(row['patch'])
        for cand in (Path(row['trace']) if row['trace'] else None, RUN_DIR / 'official' / 'traces' / f'trace_{t.instance_id}.json'):
            if cand and cand.exists():
                shutil.copy(cand, RUN_DIR / 'traces' / f'trace_{t.instance_id}.json'); break
        summary['tasks'].append(t.instance_id); summary['resolved'] += int(row['resolved'])
        print(f"[{i}/{len(sound)}] {t.instance_id} resolved={row['resolved']} calls={row['tool_calls']} "
              f"agent={row['duration_s']}s setup={row['setup_s']}s err={(row['error'] or '')[:100]}")
    n = len(summary['tasks'])
    print(f"RESOLVED {summary['resolved']}/{n}" if n else 'no tasks run')
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=1))

In [ ]:
# ---------------------------------------------------------------- bundle for download + stop server
# Official logs may contain repository snippets: keep the bundle private, never commit it (plan §1.3).
for extra in (RUN_DIR / 'official' / 'logs',):
    if extra.exists():
        shutil.copytree(extra, RUN_DIR / 'logs', dirs_exist_ok=True)
shutil.make_archive(str(Path('/kaggle/working/runs') / EXP_ID), 'zip', RUN_DIR)
print('bundle:', Path('/kaggle/working/runs') / f'{EXP_ID}.zip')
try:
    server.stop()
except Exception:  # noqa: BLE001
    pass